##Silver to Gold: Building BI ready tables

In [0]:
from pyspark.sql.types import IntegerType, StringType, StructField, StructType, TimestampType, DateType, DoubleType, FloatType
from pyspark.sql import Row
import pyspark.sql.functions as F

catalog_name = 'ecommerce'

##Products

In [0]:
df_products = spark.table(f'{catalog_name}.silver.slv_products')
df_categories = spark.table(f'{catalog_name}.silver.slv_category')
df_brands = spark.table(f'{catalog_name}.silver.slv_brands')

In [0]:
df_products.createOrReplaceTempView('v_products')
df_categories.createOrReplaceTempView('v_categories')
df_brands.createOrReplaceTempView('v_brands')

In [0]:
display(spark.sql('SELECT * FROM V_PRODUCTS'))

In [0]:
display(spark.sql('SELECT * FROM V_CATEGORIES'))

In [0]:
display(spark.sql('SELECT * FROM V_BRANDS'))

In [0]:
df_brand_categories = df_categories.join(df_brands, on=['category_code'], how='inner')
display(df_brand_categories)

In [0]:
df_brand_categories = df_brand_categories.select('brand_name', 'brand_code', 'category_name', 'category_code')
display(df_brand_categories)

In [0]:
df_joined_prd = df_products.join(df_brand_categories, on=['brand_code'], how='left')
display(df_joined_prd)

In [0]:
%sql
USE CATALOG ecommerce;

In [0]:
%sql

CREATE OR REPLACE TABLE gold.gld_dim_products AS

WITH brands_categories AS (
  SELECT
    b.brand_name,
    b.brand_code,
    c.category_name,
    c.category_code
  FROM v_brands b
  INNER JOIN v_categories c
  ON
    b.category_code = c.category_code
)
SELECT
  p.product_id,
  p.sku,
  p.category_code,
  COALESCE(bc.category_name, 'Not Available') AS category_name,
  p.brand_code,
  COALESCE(bc.brand_name, 'Not Available')   AS brand_name,
  p.color,
  p.size,
  p.material,
  p.weight_grams,
  p.length_cm,
  p.width_cm,
  p.height_cm,
  p.rating_count,
  p._source_file_name,
  p._ingested_at
FROM v_products p
LEFT JOIN brands_categories bc
  ON p.brand_code = bc.brand_code;

##Customers

In [0]:
# India states
india_region = {
    "MH": "West", "GJ": "West", "RJ": "West",
    "KA": "South", "TN": "South", "TS": "South", "AP": "South", "KL": "South",
    "UP": "North", "WB": "North", "DL": "North"
}
# Australia states
australia_region = {
    "VIC": "SouthEast", "WA": "West", "NSW": "East", "QLD": "NorthEast"
}

# United Kingdom states
uk_region = {
    "ENG": "England", "WLS": "Wales", "NIR": "Northern Ireland", "SCT": "Scotland"
}

# United States states
us_region = {
    "MA": "NorthEast", "FL": "South", "NJ": "NorthEast", "CA": "West", 
    "NY": "NorthEast", "TX": "South"
}

# UAE states
uae_region = {
    "AUH": "Abu Dhabi", "DU": "Dubai", "SHJ": "Sharjah"
}

# Singapore states
singapore_region = {
    "SG": "Singapore"
}

# Canada states
canada_region = {
    "BC": "West", "AB": "West", "ON": "East", "QC": "East", "NS": "East", "IL": "Other"
}

# Combine into a master dictionary
country_state_map = {
    "India": india_region,
    "Australia": australia_region,
    "United Kingdom": uk_region,
    "United States": us_region,
    "United Arab Emirates": uae_region,
    "Singapore": singapore_region,
    "Canada": canada_region
}  


In [0]:
rows = []
for country, state in country_state_map.items():
    for state, region in state.items():
        rows.append(Row(country=country, state=state, region=region))

rows[:10]

In [0]:
df_region_mapping = spark.createDataFrame(rows)
df_region_mapping.display()

In [0]:
df_silver_customers = spark.table(f'{catalog_name}.silver.slv_customers')
display(df_silver_customers)

In [0]:
df_gold_customers = df_silver_customers.join(df_region_mapping, on=['country', 'state'], how='left')
display(df_gold_customers)

In [0]:
df_gold_customers = df_gold_customers.fillna({'region': 'Other'})
display(df_gold_customers)

In [0]:
df_gold_customers.write.format('delta').mode('overwrite').option('mergeSchema', 'true').saveAsTable(f'{catalog_name}.gold.gld_dim_customers')

##Date/Calendar

In [0]:
df_silver_calendar = spark.table(f'{catalog_name}.silver.slv_calendar')
display(df_silver_calendar)

In [0]:
df_gold_calendar = df_silver_calendar.withColumn('date_id', F.date_format('date', 'yyyyMMdd').cast('int'))
df_gold_calendar = df_gold_calendar.withColumn('month_name', F.date_format('date', 'MMMM'))
df_gold_calendar = df_gold_calendar.withColumn('is_weekend', F.when(F.col('day_name').isin('Saturday', 'Sunday'), 1).otherwise(0))
display(df_gold_calendar)

In [0]:
desired_columns_order = ["date_id", "date", "year", "month_name", "day_name", "is_weekend", "quarter", "week_of_year", "_ingested_at", "_source_file_name"]

df_gold_calendar = df_gold_calendar.select(desired_columns_order)
display(df_gold_calendar)

In [0]:
df_gold_calendar.write.format('delta').mode('overwrite').option('mergeSchema', 'true').saveAsTable(f'{catalog_name}.gold.gld_dim_calendar')